# 06 — Baseline Sentiment Classification
**Objective 6 (part):** Train Logistic Regression, Naive Bayes, SVM, and Random Forest with TF-IDF features.  
Benchmark for the BERT model.


In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SENTIMENT_LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {l: i for i, l in enumerate(SENTIMENT_LABELS)}
RANDOM_SEED = 42
TEST_SIZE, VAL_SIZE = 0.15, 0.15

def load_cleaned():
    for name in ("cleaned_reviews.parquet", "cleaned_reviews.pkl"):
        p = DATA_DIR / name
        if p.exists() and p.stat().st_size > 0:
            return pd.read_parquet(p) if name.endswith(".parquet") else pd.read_pickle(p)
    raise FileNotFoundError("Run 02_data_cleaning first")

df = load_cleaned()
print(df.shape, df["sentiment_label"].value_counts().to_dict())


In [ ]:
# Stratified splits
X = df["review_text_ml"].values
y = df["sentiment_label"].map(LABEL2ID).values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_SEED
)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio, stratify=y_temp, random_state=RANDOM_SEED
)
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")


In [ ]:
def make_pipeline(name: str) -> Pipeline:
    vectorizer = TfidfVectorizer(
        max_features=15000, ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True,
    )
    classifiers = {
        "logistic_regression": LogisticRegression(
            max_iter=1000, C=1.0, class_weight="balanced", random_state=RANDOM_SEED
        ),
        "naive_bayes": MultinomialNB(alpha=0.5),
        "svm": LinearSVC(C=1.0, class_weight="balanced", random_state=RANDOM_SEED, dual="auto"),
        "random_forest": RandomForestClassifier(
            n_estimators=200, max_depth=30, class_weight="balanced",
            n_jobs=-1, random_state=RANDOM_SEED,
        ),
    }
    return Pipeline([("tfidf", vectorizer), ("clf", classifiers[name])])

def evaluate(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

results = []
for name in ["logistic_regression", "naive_bayes", "svm", "random_forest"]:
    print(f"\n=== Training {name} ===")
    pipe = make_pipeline(name)
    pipe.fit(X_train, y_train)
    for split, Xs, ys in [("val", X_val, y_val), ("test", X_test, y_test)]:
        preds = pipe.predict(Xs)
        metrics = evaluate(ys, preds)
        print(f"  {split}: acc={metrics['accuracy']:.4f}  f1_macro={metrics['f1_macro']:.4f}")
        if split == "test":
            print(classification_report(ys, preds, target_names=SENTIMENT_LABELS, zero_division=0))
            results.append({"model": name, **metrics})
    joblib.dump(pipe, MODELS_DIR / f"baseline_{name}.joblib")

results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
display(results_df)
results_df.to_csv(MODELS_DIR / "baseline_results.csv", index=False)
